# PPG-to-Motion Dataset Validation
Plots one processed segment from each data source (PPG-DaLiA and VSM Pre-OP)
showing the raw signal, ACC magnitude, differential image, and STFT image.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from ppg_to_motion.io.ppg_dalia import ppg_dalia_generator
from ppg_to_motion.io.vsm_preop import vsm_generator
from ppg_to_motion.preprocessing.signal_image_2d import (
    generate_differential_signal_image,
    generate_stft_features,
    normalize_multichannel_image,
)

DALIA_ROOT = r'C:\datasets\ppg+dalia'
VSM_ROOT   = r'C:\datasets\Multi-Arrhythmia Hospital Pre-OP Data\Multi-Arrhythmia Hospital Pre-OP Data\Raw Data'
VSM_LOG    = r'C:\datasets\Multi-Arrhythmia Hospital Pre-OP Data\Multi-Arrhythmia Hospital Pre-OP Data\Multi-Arrhythmia Patients (Hospital pre-op).xlsx'

FS       = 100.0   # all signals stored at 100 Hz
N_FFT    = 128
HOP      = 16
FREQ_MAX = FS / 2  # 50 Hz


def compute_images(signal):
    diff = normalize_multichannel_image(
        generate_differential_signal_image(signal), method='zscore'
    )
    stft = generate_stft_features(signal, n_fft=N_FFT, hop_length=HOP)
    return diff, stft


def plot_sample(sample, title, color):
    signal = np.asarray(sample['signal'], dtype=np.float32)
    acc    = np.asarray(sample['acc'],    dtype=np.float32)
    t      = np.arange(len(signal)) / FS

    diff, stft = compute_images(signal)

    fig = plt.figure(figsize=(15, 10))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.35)

    # --- Row 0: PPG + ACC ---
    ax = fig.add_subplot(gs[0, :2])
    ax.plot(t, signal, color=color, lw=0.8)
    ax.set(title='PPG signal (100 Hz)', xlabel='Time (s)', ylabel='Amplitude')
    ax.grid(True, alpha=0.3)

    ax = fig.add_subplot(gs[0, 2])
    ax.plot(t, acc, color='dimgrey', lw=0.8)
    ax.set(title='ACC magnitude', xlabel='Time (s)', ylabel='m/s²')
    ax.grid(True, alpha=0.3)

    # --- Row 1: differential image channels ---
    for i, ch_name in enumerate(['x (raw)', 'dx/dt', 'd²x/dt²']):
        ax = fig.add_subplot(gs[1, i])
        ax.plot(t, diff[i], lw=0.7, color=color)
        ax.set(title=f'diff ch{i}: {ch_name}', xlabel='Time (s)')
        ax.grid(True, alpha=0.3)

    # --- Row 2: STFT image channels ---
    extent = [0, t[-1], 0, FREQ_MAX]
    stft_titles = ['log|X| (z-scored)', 'cos φ (z-scored)', 'sin φ (z-scored)']
    for i, ch_name in enumerate(stft_titles):
        ax = fig.add_subplot(gs[2, i])
        im = ax.imshow(
            stft[i], aspect='auto', origin='lower',
            extent=extent, cmap='RdBu_r', vmin=-2, vmax=2,
        )
        ax.set(title=f'STFT ch{i}: {ch_name}', xlabel='Time (s)', ylabel='Freq (Hz)')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(title, fontsize=13, y=1.01)
    plt.show()

## 1  PPG-DaLiA — first 30-second segment

In [ ]:
dalia_sample = next(ppg_dalia_generator(DALIA_ROOT))

print(f"ID            : {dalia_sample['ID']}")
print(f"activity      : {dalia_sample.get('activity', 'N/A')}")
print(f"label (HR)    : {dalia_sample.get('label', 'N/A')}")
print(f"sampling_rate : {dalia_sample['sampling_rate']} Hz")
print(f"signal shape  : {np.asarray(dalia_sample['signal']).shape}")
print(f"acc shape     : {np.asarray(dalia_sample['acc']).shape}")

In [ ]:
label_str    = f"HR={dalia_sample['label']:.1f} BPM" if 'label' in dalia_sample else ''
activity_str = dalia_sample.get('activity', '')
title = f"PPG-DaLiA | ID={dalia_sample['ID']} | {activity_str} | {label_str}"

plot_sample(dalia_sample, title, color='darkorange')

## 2  VSM Pre-OP — first 30-second segment

In [ ]:
vsm_sample = next(vsm_generator(VSM_ROOT, VSM_LOG))

print(f"ID            : {vsm_sample['ID']}")
print(f"rhythm_label  : {vsm_sample.get('rhythm_label', 'N/A')}")
print(f"sampling_rate : {vsm_sample['sampling_rate']} Hz")
print(f"signal shape  : {np.asarray(vsm_sample['signal']).shape}")
print(f"acc shape     : {np.asarray(vsm_sample['acc']).shape}")

In [ ]:
title = f"VSM Pre-OP | ID={vsm_sample['ID']} | {vsm_sample.get('rhythm_label', '')}"

plot_sample(vsm_sample, title, color='steelblue')